In [1]:
import pandas as pd
portfolio = pd.read_csv("25_Portfolios_5x5.csv", skiprows = 15)

Import and read "25_Portfolios_5x5.csv". Skip the headlines.

In [2]:
portfolio = portfolio.rename(columns = {"Unnamed: 0": "Date"})
portfolio = portfolio[portfolio["Date"].astype(str).str.isnumeric()]
portfolio = portfolio[portfolio["Date"].astype(str).str.len() == 6]
portfolio = portfolio.drop_duplicates(subset = "Date", keep = "first")
portfolio["Date"] = pd.to_datetime(portfolio["Date"], format = "%Y%m")
portfolio = portfolio.rename(columns = {"SMALL LoBM": "ME1 BM1"})
portfolio = portfolio.rename(columns = {"SMALL HiBM": "ME1 BM5"})
portfolio = portfolio.rename(columns = {"BIG LoBM": "ME5 BM1"})
portfolio = portfolio.rename(columns = {"BIG HiBM": "ME5 BM5"})

Clean, rename, and change datatype for "25_Portfolios_5x5.csv". 

In [3]:
factor = pd.read_csv("F-F_Research_Data_Factors.CSV", skiprows = 3)
factor = factor.rename(columns = {"Unnamed: 0": "Date"})
factor = factor.rename(columns = {"Mkt-RF": "RM-RF"})
factor = factor[factor["Date"].astype(str).str.isnumeric()]
factor = factor[factor["Date"].astype(str).str.len() == 6]
factor["Date"] = pd.to_datetime(factor["Date"], format = "%Y%m")

Import and read "F-F_Research_Data_Factors.CSV". Rename and change datatype for the columns. 

In [4]:
import numpy as np
factor["RM-RF"] = pd.to_numeric(factor["RM-RF"], errors = "coerce")
mean = factor["RM-RF"].mean()
std = factor["RM-RF"].std() 
t = mean / (std / np.sqrt(len(factor["RM-RF"])))
skew = factor["RM-RF"].skew()
kurt = factor["RM-RF"].kurtosis() + 3
t = t.round(3)
skew = skew.round(3)
kurt = kurt.round(3)
print(f"mean: {mean:.3f}")
print(f"std: {std:.3f}")
print("t:", t)
print("Skewness:", skew)
print("Kurtosis:", kurt)
print(" ")

rmrf = factor["RM-RF"].dropna()

autocorr_1 = rmrf.autocorr(lag=1)
autocorr_2 = rmrf.autocorr(lag=2)
autocorr_12 = rmrf.autocorr(lag=12)

print("\nAutocorrelation:")
print("1-month autocorr: ", round(autocorr_1, 3))
print("2-month autocorr:", round(autocorr_2, 3))
print("12-month autocorr:", round(autocorr_12, 3))

from scipy.stats import normaltest
if t > 1.96:
    print("Reject H0, the mean is significantly different from 0.")
else:
    print("Fail to reject H0, the mean is not significantly different from 0.")

stat, p = normaltest(factor["RM-RF"])
print("p-value:", p)
if p < 0.05:
    print("Reject H0, the excess return does not follow a normal distribution.")
else:
    print("Fail to reject H0, the excess return follos a normal distribution.")

mean: 0.692
std: 5.314
t: 4.494
Skewness: 0.152
Kurtosis: 10.398
 

Autocorrelation:
1-month autocorr:  0.093
2-month autocorr: -0.023
12-month autocorr: 0.004
Reject H0, the mean is significantly different from 0.
p-value: 2.873977185454245e-41
Reject H0, the excess return does not follow a normal distribution.


Do the stats calculation for "F-F_Research_Data_Factors.CSV". Test to see whether it's normal or not. 

In [5]:
portfolio_cols = [c for c in portfolio.columns if c.startswith("ME") and "BM" in c]
portfolio[portfolio_cols] = portfolio[portfolio_cols].apply(pd.to_numeric, errors = "coerce")
def summary_stats(series):
    s = series.dropna()
    mean = s.mean()
    std = s.std()
    t = mean / (std / np.sqrt(len(s)))
    skew = s.skew()
    kurt = s.kurtosis() + 3
    return mean, std, t, skew, kurt

results = []

for col in portfolio_cols:
    mean, std, t, skew, kurt = summary_stats(portfolio[col])
    results.append([col, mean, std, t, skew, kurt])

stats_table = pd.DataFrame(results, columns = ["Portfolio", "Mean", "Std", "t-stat", "Skewness", "Kurtosis"])
print(stats_table.round(3))

   Portfolio   Mean     Std  t-stat  Skewness  Kurtosis
0    ME1 BM1  0.842  11.897   2.444     3.085    36.154
1    ME1 BM2  0.932   9.640   3.335     2.909    37.048
2    ME1 BM3  1.251   8.831   4.888     2.099    21.558
3    ME1 BM4  1.396   8.259   5.831     2.709    34.998
4    ME1 BM5  1.610   9.143   6.076     2.851    31.325
5    ME2 BM1  0.915   7.948   3.975     0.702    11.918
6    ME2 BM2  1.196   7.460   5.532     1.407    19.999
7    ME2 BM3  1.231   7.204   5.899     1.680    22.409
8    ME2 BM4  1.303   7.419   6.059     2.061    25.844
9    ME2 BM5  1.498   8.655   5.973     1.702    20.954
10   ME3 BM1  0.987   7.330   4.647     0.792    11.965
11   ME3 BM2  1.171   6.465   6.251     0.098     9.015
12   ME3 BM3  1.182   6.447   6.328     0.966    15.984
13   ME3 BM4  1.268   6.916   6.328     1.740    23.375
14   ME3 BM5  1.375   8.358   5.679     1.564    19.001
15   ME4 BM1  0.999   6.176   5.582    -0.195     6.882
16   ME4 BM2  1.047   6.081   5.939     0.624   

In [6]:
stats_table["ME"] = stats_table["Portfolio"].str.extract(r"ME(\d)").astype(int)
stats_table["BM"] = stats_table["Portfolio"].str.extract(r"BM(\d)").astype(int)

def show_table(metric):
    table = stats_table.pivot(index = "ME", columns = "BM", values = metric)
    print(f"\n{metric}:")
    print(table.round(3))

show_table("Mean")
show_table("Std")
show_table("t-stat")
show_table("Skewness")
show_table("Kurtosis")


Mean:
BM      1      2      3      4      5
ME                                   
1   0.842  0.932  1.251  1.396  1.610
2   0.915  1.196  1.231  1.303  1.498
3   0.987  1.171  1.182  1.268  1.375
4   0.999  1.047  1.108  1.236  1.297
5   0.963  0.920  0.978  0.940  1.235

Std:
BM       1      2      3      4      5
ME                                    
1   11.897  9.640  8.831  8.259  9.143
2    7.948  7.460  7.204  7.419  8.655
3    7.330  6.465  6.447  6.916  8.358
4    6.176  6.081  6.281  6.805  8.545
5    5.326  5.243  5.564  6.520  8.445

t-stat:
BM      1      2      3      4      5
ME                                   
1   2.444  3.335  4.888  5.831  6.076
2   3.975  5.532  5.899  6.059  5.973
3   4.647  6.251  6.328  6.328  5.679
4   5.582  5.939  6.090  6.266  5.237
5   6.243  6.055  6.068  4.977  5.046

Skewness:
BM      1      2      3      4      5
ME                                   
1   3.085  2.909  2.099  2.709  2.851
2   0.702  1.407  1.680  2.061  1.702
3   0.792 

In [9]:
import statsmodels.api as sm

factor["RF"] = pd.to_numeric(factor["RF"], errors = "coerce")
data_merge = pd.merge(portfolio, factor[["Date", "RM-RF", "RF"]], on = "Date", how = "inner")

reg_results = []

for col in portfolio_cols:
    y = data_merge[col]
    X = sm.add_constant(data_merge["RM-RF"])
    model = sm.OLS(y, X, missing = "drop").fit()
    robust = model.get_robustcov_results(cov_type = "HC0")
    se = np.sqrt(model.ssr / model.df_resid)
    reg_results.append([
        col,
        model.params["const"],
        model.params["RM-RF"],
        robust.tvalues[0],
        robust.tvalues[1],
        model.rsquared,
        se
    ])

reg_table = pd.DataFrame(
    reg_results,
    columns = ["Portfolio", "Alpha", "Beta", "t(Alpha)", "t(Beta)", "R2", "s(e)"]
)

print(reg_table.round(3))

reg_table["ME"] = reg_table["Portfolio"].str.extract(r"ME(\d)").astype(int)
reg_table["BM"] = reg_table["Portfolio"].str.extract(r"BM(\d)").astype(int)

def show_reg(metric):
    table = reg_table.pivot(index = "ME", columns = "BM", values = metric)
    print(f"\n{metric}:")
    print(table.round(3))

show_reg("Alpha")
show_reg("Beta")
show_reg("t(Alpha)")
show_reg("t(Beta)")
show_reg("R2")
show_reg("s(e)")

   Portfolio  Alpha   Beta  t(Alpha)  t(Beta)     R2   s(e)
0    ME1 BM1 -0.264  1.599    -1.282   12.030  0.510  8.328
1    ME1 BM2 -0.030  1.390    -0.194   15.372  0.587  6.198
2    ME1 BM3  0.312  1.356     2.303   20.485  0.666  5.105
3    ME1 BM4  0.523  1.261     4.269   18.086  0.659  4.827
4    ME1 BM5  0.670  1.357     4.694   17.136  0.622  5.622
5    ME2 BM1  0.039  1.267     0.336   25.709  0.717  4.227
6    ME2 BM2  0.346  1.229     3.647   25.871  0.766  3.611
7    ME2 BM3  0.403  1.197     4.570   27.222  0.780  3.378
8    ME2 BM4  0.464  1.212     4.821   23.730  0.754  3.681
9    ME2 BM5  0.547  1.373     4.367   22.614  0.711  4.654
10   ME3 BM1  0.128  1.242     1.477   38.014  0.810  3.193
11   ME3 BM2  0.390  1.128     5.643   54.737  0.860  2.420
12   ME3 BM3  0.409  1.116     5.799   36.991  0.847  2.523
13   ME3 BM4  0.460  1.168     5.622   25.555  0.805  3.055
14   ME3 BM5  0.439  1.354     3.781   27.330  0.741  4.259
15   ME4 BM1  0.248  1.085     3.873   5